# DATA WRANGLING

In [137]:
import pandas as pd

jobs_df = pd.read_csv("dataset/jobs.csv")
skills_df = pd.read_csv("dataset/skills.csv")
resume_df = pd.read_csv("dataset/resume.csv")

In [138]:
jobs_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 324 entries, 0 to 323
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Job Title              324 non-null    object
 1   Category               324 non-null    object
 2   Education Requirement  324 non-null    object
 3   Experience Years       324 non-null    int64 
 4   Required Skills        324 non-null    object
 5   Salary Range           324 non-null    object
dtypes: int64(1), object(5)
memory usage: 15.3+ KB


In [139]:
skills_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Skill Name  120 non-null    object
 1   Category    120 non-null    object
dtypes: object(2)
memory usage: 2.0+ KB


In [140]:
resume_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Resume ID         10000 non-null  object
 1   Resume Text       10000 non-null  object
 2   Education         10000 non-null  object
 3   Experience Years  10000 non-null  int64 
 4   Skills            10000 non-null  object
 5   Job Role          10000 non-null  object
 6   Category          10000 non-null  object
dtypes: int64(1), object(6)
memory usage: 547.0+ KB


In [141]:
# Normalize semua text (spasi, huruf kapital/kecil)
jobs_df = jobs_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
skills_df = skills_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
resume_df = resume_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

C:\Users\joyce\AppData\Local\Temp\ipykernel_13256\1157588541.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  jobs_df = jobs_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
C:\Users\joyce\AppData\Local\Temp\ipykernel_13256\1157588541.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  skills_df = skills_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
C:\Users\joyce\AppData\Local\Temp\ipykernel_13256\1157588541.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  resume_df = resume_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


In [142]:
# Merapikan nama kolom
jobs_df.columns = jobs_df.columns.str.lower().str.replace(" ", "_")
skills_df.columns = skills_df.columns.str.lower().str.replace(" ", "_")
resume_df.columns = resume_df.columns.str.lower().str.replace(" ", "_")

In [143]:
# Cleaning skills (dari resume.csv)
def clean_skills(text):
    text = text.replace("[","").replace("]","").replace("'","")
    skills = text.split(",")
    return [s.strip() for s in skills if s.strip() != ""]

resume_df['skills'] = resume_df['skills'].apply(clean_skills)

In [144]:
# Cleaning skills (dari jobs.csv)
jobs_df['required_skills'] = jobs_df['required_skills'].apply(clean_skills)

In [145]:
# Samakan nama kolom untuk job di jobs.csv dan resume.csv
jobs_df['job_title'] = jobs_df['job_title'].str.strip()
resume_df['job_role'] = resume_df['job_role'].str.strip()

In [146]:
# Cek udah cocok belum (kalau kosong -> job sudah match)
set(resume_df['job_role']) - set(jobs_df['job_title'])

set()

In [147]:
# Normalisasi singkatan jurusan
import re
def clean_education_v2(text):
    text = text.lower()
    
    mapping = {
        "it": "information technology",
        "rn": "nursing",
        "mba": "business administration",
        "mpa": "public administration",
        "hr": "human resources",
        "cpa": "accounting",
        "qa": "quality assurance",
        "ai": "artificial intelligence",
        "pt": "physical therapy"
    }
    
    for k, v in mapping.items():
        text = re.sub(rf'\b{k}\b', v, text)
    
    return text.strip()

resume_df['education_clean_v2'] = resume_df['education'].apply(clean_education_v2)
jobs_df['education_clean_v2'] = jobs_df['education_requirement'].apply(clean_education_v2)

In [148]:
# Drop kolom lama education dan ganti dengan yang baru
resume_df = resume_df.drop(columns=['education'])
resume_df = resume_df.rename(columns={'education_clean_v2': 'education'})

jobs_df = jobs_df.drop(columns=['education_requirement'])
jobs_df = jobs_df.rename(columns={'education_clean_v2': 'education_requirement'})

In [149]:
# Cek apakah sudah terganti
resume_df.head(10)

,resume_id,resume_text,experience_years,skills,job_role,category,education
0,r000000,education: bachelor's in computer science expe...,2,[market research|maven|java|rest api|spring bo...,java backend developer,technology,bachelor's in computer science
1,r000001,education: master's in microbiology experience...,3,[agile|data analysis|precision|problem solving...,microbiologist,science & research,master's in microbiology
2,r000002,education: apprenticeship experience: 2 years ...,2,[react|customer service|technical knowledge|pl...,plumber,skilled trades,apprenticeship
3,r000003,education: bachelor's in computer science expe...,0,[mysql|query optimization|symfony|terraform|ph...,php developer,technology,bachelor's in computer science
4,r000004,education: bachelor's in design experience: 3 ...,3,[problem solving|docker|creativity|scrum|ansib...,art director,creative & design,bachelor's in design
5,r000005,education: administrative certification experi...,2,[ci/cd|problem solving|database design|pytorch...,executive assistant,administration,administrative certification
6,r000006,education: bachelor's in computer science expe...,3,[distributed systems|cql|cassandra|performance...,cassandra developer,technology,bachelor's in computer science
7,r000007,education: bachelor's in it experience: 1 year...,1,[performance tuning|sql|etl|data integration|j...,informatica developer,data & analytics,bachelor's in information technology
8,r000008,education: pt license experience: 4 years skil...,4,[communication|swift|anatomy knowledge|exercis...,physical therapist,healthcare,physical therapy license
9,r000009,education: bachelor's in computer science expe...,5,[postgresql|distributed systems|kanban|phoenix...,elixir developer,technology,bachelor's in computer science


In [150]:
jobs_df.head(10)

,job_title,category,experience_years,required_skills,salary_range,education_requirement
0,software engineer,technology,2,[python|java|c++|git|software design|problem s...,80-150k,bachelor's in computer science|bachelor's in e...
1,full stack developer,technology,2,[javascript|react|node.js|html/css|database|git],75-140k,diploma in information technology|bachelor's i...
2,frontend developer,technology,1,[javascript|react|vue.js|css|html|ui/ux design],70-130k,diploma in information technology|bachelor's i...
3,backend developer,technology,2,[python|java|node.js|database design|api devel...,75-135k,bachelor's in computer science
4,devops engineer,technology,3,[docker|kubernetes|aws|linux|ci/cd|scripting],85-150k,bachelor's in computer science|information tec...
5,cloud architect,technology,5,[aws|azure|gcp|system design|networking|security],120-200k,bachelor's in computer science|cloud certifica...
6,data scientist,technology,2,[python|machine learning|statistics|tensorflow...,90-160k,master's in data science|bachelor's in mathema...
7,machine learning engineer,technology,3,[python|tensorflow|pytorch|deep learning|stati...,100-180k,master's in computer science|master's in artif...
8,data engineer,technology,2,[python|spark|hadoop|sql|data warehousing|etl],85-155k,bachelor's in computer science
9,database administrator,technology,2,[sql|mongodb|database optimization|linux|backu...,70-130k,bachelor's in information technology|informati...


In [151]:
print(jobs_df['required_skills'].iloc[0])
print(type(jobs_df['required_skills'].iloc[0]))

['python|java|c++|git|software design|problem solving']
<class 'list'>


In [152]:
# Cleaning skills
def clean_skills(text):
    import pandas as pd
    
    if pd.isna(text):
        return []
    
    # kalau list
    if isinstance(text, list):
        # kalau list cuma 1 elemen → ambil isinya
        if len(text) == 1:
            text = text[0]
        else:
            return [str(s).strip().lower() for s in text]
    
    # sekarang pasti string
    text = str(text).lower()
    
    text = text.replace("[","").replace("]","")
    text = text.replace("|", ",")
    
    skills = text.split(",")
    
    return [s.strip() for s in skills if s.strip() != ""]

In [153]:
jobs_df['required_skills'] = jobs_df['required_skills'].apply(clean_skills)
resume_df['skills'] = resume_df['skills'].apply(clean_skills)

In [156]:
# Cek apakah sudah berubah jadi list di bagian required skills & skills
jobs_df.head()

,job_title,category,experience_years,required_skills,salary_range,education_requirement
0,software engineer,technology,2,"[python, java, c++, git, software design, prob...",80-150k,bachelor's in computer science|bachelor's in e...
1,full stack developer,technology,2,"[javascript, react, node.js, html/css, databas...",75-140k,diploma in information technology|bachelor's i...
2,frontend developer,technology,1,"[javascript, react, vue.js, css, html, ui/ux d...",70-130k,diploma in information technology|bachelor's i...
3,backend developer,technology,2,"[python, java, node.js, database design, api d...",75-135k,bachelor's in computer science
4,devops engineer,technology,3,"[docker, kubernetes, aws, linux, ci/cd, script...",85-150k,bachelor's in computer science|information tec...


In [157]:
resume_df.head()

,resume_id,resume_text,experience_years,skills,job_role,category,education
0,r000000,education: bachelor's in computer science expe...,2,"[market research, maven, java, rest api, sprin...",java backend developer,technology,bachelor's in computer science
1,r000001,education: master's in microbiology experience...,3,"[agile, data analysis, precision, problem solv...",microbiologist,science & research,master's in microbiology
2,r000002,education: apprenticeship experience: 2 years ...,2,"[react, customer service, technical knowledge,...",plumber,skilled trades,apprenticeship
3,r000003,education: bachelor's in computer science expe...,0,"[mysql, query optimization, symfony, terraform...",php developer,technology,bachelor's in computer science
4,r000004,education: bachelor's in design experience: 3 ...,3,"[problem solving, docker, creativity, scrum, a...",art director,creative & design,bachelor's in design


In [158]:
# Hapus kolom tidak perlu
jobs_df.drop(columns=['experience_years', 'salary_range'], inplace=True)
resume_df.drop(columns=['experience_years'], inplace=True)

In [159]:
# Cek lagi tabel final
jobs_df.head()

,job_title,category,required_skills,education_requirement
0,software engineer,technology,"[python, java, c++, git, software design, prob...",bachelor's in computer science|bachelor's in e...
1,full stack developer,technology,"[javascript, react, node.js, html/css, databas...",diploma in information technology|bachelor's i...
2,frontend developer,technology,"[javascript, react, vue.js, css, html, ui/ux d...",diploma in information technology|bachelor's i...
3,backend developer,technology,"[python, java, node.js, database design, api d...",bachelor's in computer science
4,devops engineer,technology,"[docker, kubernetes, aws, linux, ci/cd, script...",bachelor's in computer science|information tec...


In [160]:
resume_df.head()

,resume_id,resume_text,skills,job_role,category,education
0,r000000,education: bachelor's in computer science expe...,"[market research, maven, java, rest api, sprin...",java backend developer,technology,bachelor's in computer science
1,r000001,education: master's in microbiology experience...,"[agile, data analysis, precision, problem solv...",microbiologist,science & research,master's in microbiology
2,r000002,education: apprenticeship experience: 2 years ...,"[react, customer service, technical knowledge,...",plumber,skilled trades,apprenticeship
3,r000003,education: bachelor's in computer science expe...,"[mysql, query optimization, symfony, terraform...",php developer,technology,bachelor's in computer science
4,r000004,education: bachelor's in design experience: 3 ...,"[problem solving, docker, creativity, scrum, a...",art director,creative & design,bachelor's in design


In [161]:
# Sebelum dimerge, pastikan nama kolom sama
jobs_df['job_title'] = jobs_df['job_title'].str.strip().str.lower()
resume_df['job_role'] = resume_df['job_role'].str.strip().str.lower()

In [163]:
# Cek apakah sudah match apa belum (kosong berarti match)
set(resume_df['job_role']) - set(jobs_df['job_title'])

set()

In [164]:
# Merge tabel
merged_df = resume_df.merge(
    jobs_df,
    left_on='job_role',
    right_on='job_title',
    how='inner'
)

In [165]:
merged_df.head()

,resume_id,resume_text,skills,job_role,category_x,education,job_title,category_y,required_skills,education_requirement
0,r000000,education: bachelor's in computer science expe...,"[market research, maven, java, rest api, sprin...",java backend developer,technology,bachelor's in computer science,java backend developer,technology,"[java, spring boot, sql, rest api, microservic...",bachelor's in computer science
1,r000001,education: master's in microbiology experience...,"[agile, data analysis, precision, problem solv...",microbiologist,science & research,master's in microbiology,microbiologist,science & research,"[microbiology, laboratory work, research, data...",master's in microbiology
2,r000002,education: apprenticeship experience: 2 years ...,"[react, customer service, technical knowledge,...",plumber,skilled trades,apprenticeship,plumber,skilled trades,"[plumbing systems, safety, problem solving, te...",trade school|apprenticeship
3,r000003,education: bachelor's in computer science expe...,"[mysql, query optimization, symfony, terraform...",php developer,technology,bachelor's in computer science,php developer,technology,"[php, laravel, symfony, mysql, api development...",bachelor's in computer science
4,r000004,education: bachelor's in design experience: 3 ...,"[problem solving, docker, creativity, scrum, a...",art director,creative & design,bachelor's in design,art director,creative & design,"[artistic direction, visual communication, tea...",bachelor's in art|bachelor's in design


In [166]:
len(merged_df)

10000

In [168]:
# Pakai kolom category dari 1 dataset saja (supaya tidak jadi x dan y)
merged_df = merged_df.drop(columns=['category_x'])
merged_df = merged_df.rename(columns={'category_y': 'category'})

In [170]:
# Pakai kolom job role saja untuk jadi label (job_title dihapus)
merged_df = merged_df.drop(columns=['job_title'])

In [171]:
# Cek tabel final
merged_df.head()

,resume_id,resume_text,skills,job_role,education,category,required_skills,education_requirement
0,r000000,education: bachelor's in computer science expe...,"[market research, maven, java, rest api, sprin...",java backend developer,bachelor's in computer science,technology,"[java, spring boot, sql, rest api, microservic...",bachelor's in computer science
1,r000001,education: master's in microbiology experience...,"[agile, data analysis, precision, problem solv...",microbiologist,master's in microbiology,science & research,"[microbiology, laboratory work, research, data...",master's in microbiology
2,r000002,education: apprenticeship experience: 2 years ...,"[react, customer service, technical knowledge,...",plumber,apprenticeship,skilled trades,"[plumbing systems, safety, problem solving, te...",trade school|apprenticeship
3,r000003,education: bachelor's in computer science expe...,"[mysql, query optimization, symfony, terraform...",php developer,bachelor's in computer science,technology,"[php, laravel, symfony, mysql, api development...",bachelor's in computer science
4,r000004,education: bachelor's in design experience: 3 ...,"[problem solving, docker, creativity, scrum, a...",art director,bachelor's in design,creative & design,"[artistic direction, visual communication, tea...",bachelor's in art|bachelor's in design


In [172]:
merged_df.to_csv("cleaned_dataset.csv", index=False)